# Simple Monte-Carlo Search

## Obiettivo

La **Simple Monte-Carlo Search** valuta ogni azione disponibile eseguendo numerosi
rollout casuali. L'azione con il rendimento medio più alto viene scelta.

In questo esempio, un agente deve decidere quanto avanzare in un piccolo gioco numerico.

In [1]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

In [2]:
TARGET = 10
ACTIONS = [1, 2, 3]

def step(state, action):
    """Applica un'azione e restituisce nuovo stato, ricompensa e termine."""
    next_state = state + action

    if next_state == TARGET:
        return next_state, 1.0, True

    if next_state > TARGET:
        return next_state, -1.0, True

    return next_state, 0.0, False

In [3]:
def random_rollout(initial_state, first_action):
    """Esegue una simulazione casuale dopo una prima azione fissata."""
    state, reward, done = step(initial_state, first_action)
    total_reward = reward

    while not done:
        action = random.choice(ACTIONS)
        state, reward, done = step(state, action)
        total_reward += reward

    return total_reward

In [4]:
def monte_carlo_search(state, simulations_per_action=1000):
    """Valuta ogni azione tramite numerosi rollout casuali."""
    estimates = {}

    for action in ACTIONS:
        returns = [
            random_rollout(state, action)
            for _ in range(simulations_per_action)
        ]
        estimates[action] = np.mean(returns)

    best_action = max(estimates, key=estimates.get)
    return best_action, estimates

## Scelta dell'azione da uno stato iniziale

In [5]:
state = 0
best_action, estimates = monte_carlo_search(
    state,
    simulations_per_action=2000,
)

print("Valore medio stimato per azione:")
for action, value in estimates.items():
    print(f"Azione {action}: {value:.3f}")

print("\nAzione scelta:", best_action)

Valore medio stimato per azione:
Azione 1: -0.038
Azione 2: 0.025
Azione 3: 0.007

Azione scelta: 2


## Uso ripetuto della ricerca

A ogni stato viene eseguita una nuova ricerca Monte-Carlo.

In [6]:
state = 0
trajectory = [state]

while state < TARGET:
    action, estimates = monte_carlo_search(
        state,
        simulations_per_action=500,
    )

    state, reward, done = step(state, action)
    trajectory.append(state)

    print(
        f"Stato precedente={trajectory[-2]}, "
        f"azione={action}, nuovo stato={state}, ricompensa={reward}"
    )

    if done:
        break

print("\nTraiettoria:", trajectory)

Stato precedente=0, azione=2, nuovo stato=2, ricompensa=0.0
Stato precedente=2, azione=2, nuovo stato=4, ricompensa=0.0
Stato precedente=4, azione=3, nuovo stato=7, ricompensa=0.0
Stato precedente=7, azione=3, nuovo stato=10, ricompensa=1.0

Traiettoria: [0, 2, 4, 7, 10]


## Osservazioni

La ricerca non costruisce un albero persistente. Ogni azione viene valutata
separatamente tramite rollout completi. Questo approccio è semplice, ma non riutilizza
in modo efficiente le simulazioni tra decisioni diverse.